# StatsBomb Passing Compass — exploration

2022 World Cup Final (Argentina vs France), match_id `3869685`.
Raw events fetched to `data/raw/statsbomb/events_3869685.json` (gitignored).
Sanity-checks here informed `scripts/generate_passing_compass_data.py`.

In [ ]:
import json
import pandas as pd

with open('../data/raw/statsbomb/events_3869685.json') as f:
    events = json.load(f)

len(events)

In [ ]:
passes = [e for e in events if e['type']['name'] == 'Pass']
df = pd.DataFrame([{
    'period': e['period'],
    'minute': e['minute'],
    'team': e['team']['name'],
    'player': e['player']['name'],
    'x0': e['location'][0], 'y0': e['location'][1],
    'x1': e['pass']['end_location'][0], 'y1': e['pass']['end_location'][1],
    'outcome': e['pass'].get('outcome', {}).get('name', 'Complete'),
    'possession': e['possession'],
} for e in passes])
df.head()

## Confirming StatsBomb's coordinate convention

Sanity check: does each team's own data already treat their attacking direction as `+x`,
or does it flip at halftime (fixed pitch/camera frame)? If shots cluster near high `x`
for BOTH teams in EVERY period, StatsBomb has already normalized per-team direction and
no goalkeeper-based flip is needed.

In [ ]:
shots = [e for e in events if e['type']['name'] == 'Shot']
shot_df = pd.DataFrame([{
    'team': e['team']['name'], 'period': e['period'], 'x': e['location'][0]
} for e in shots if 'location' in e])
shot_df.groupby(['team', 'period'])['x'].agg(['mean', 'count'])

Result: both teams average ~90-110 (attacking end) in every period, including period 2 —
confirming no flip is needed. See `[[project_statsbomb_passing_compass]]` memory update.

In [ ]:
df['dx'] = df['x1'] - df['x0']
df['dy'] = df['y1'] - df['y0']
df.groupby('team')[['dx', 'dy']].describe()

In [ ]:
with open('../public/data/statsbomb/wc2022-final-passing.json') as f:
    generated = json.load(f)

print(generated['teams'])
print(len(generated['passes']), 'passes,', len(generated['triangles']), 'triangles')